In [2]:
import pyranges as pr
import gzip
import pandas as pd
import numpy as np
from scipy.sparse import dok_matrix

from hicexplorer import hicConvertFormat

In [3]:
path_interactions = "/home/joachim/data/MHH/epigenetic_graph_analyzer/phic/GSM6505208_Md_CapHiC_Merged_All-Tr_2ndBioRep_Chicago_output_washU_text.txt.gz"
path_output_2d = "/home/joachim/data/MHH/epigenetic_graph_analyzer/phic/GSM6505208_Md_CapHiC_Merged_All-Tr_2ndBioRep_Chicago_output_washU_text.csv"
path_output_cool = "/home/joachim/data/MHH/epigenetic_graph_analyzer/phic/GSM6505208_Md_CapHiC_Merged_All-Tr_2ndBioRep_Chicago_output_washU_text.cool"
path_mm10_chromosome_size = "/home/joachim/data/MHH/epigenetic_graph_analyzer/mm10.chrom.sizes"
path_promoter = "/home/joachim/data/MHH/epigenetic_graph_analyzer/mm10_promoter.bed"


In [4]:
pd_interactions = pd.read_table(path_interactions, header=None, sep='\t|,', compression='gzip')

In [5]:
pd_interactions

,0,1,2,3,4,5,6
0,chr1,3100832,3102475,chr1,88220832,88222715,20.75
1,chr1,3100832,3102475,chr1,88239615,88243256,7.79
2,chr1,3100832,3102475,chr1,88265935,88274374,7.79
3,chr1,3100832,3102475,chr1,88278578,88280173,14.24
4,chr1,3100832,3102475,chr10,53085931,53086971,5.24
...,...,...,...,...,...,...,...
324460,chrY,1304307,1309275,chr9,8775476,8777570,7.28
324461,chrY,1304307,1309275,chrY,790667,792380,5.28
324462,chrY,1304307,1309275,chrY,981956,982979,7.06
324463,chrY,1304307,1309275,chrY,1017923,1018908,5.10


In [6]:
pd_interactions.to_csv(path_output_2d, index=False, header=False, sep='\t')

In [22]:
args = "--matrices {} --outFileName {} --inputFormat 2D-text --outputFormat cool -r 10000 --chromosomeSizes {}".format(path_output_2d, path_output_cool, path_mm10_chromosome_size).split()
hicConvertFormat.main(args)

ERROR:hicmatrix.HiCMatrix:Index error
Traceback (most recent call last):
  File "/home/joachim/miniconda3/envs/ega/lib/python3.10/site-packages/hicmatrix/HiCMatrix.py", line 262, in getRegionBinRange
    endbin = sorted(self.interval_trees[chrname][endpos:endpos + 1])[0].data
IndexError: list index out of range
ERROR:hicmatrix.HiCMatrix:Index error
Traceback (most recent call last):
  File "/home/joachim/miniconda3/envs/ega/lib/python3.10/site-packages/hicmatrix/HiCMatrix.py", line 262, in getRegionBinRange
    endbin = sorted(self.interval_trees[chrname][endpos:endpos + 1])[0].data
IndexError: list index out of range
ERROR:hicmatrix.HiCMatrix:Index error
Traceback (most recent call last):
  File "/home/joachim/miniconda3/envs/ega/lib/python3.10/site-packages/hicmatrix/HiCMatrix.py", line 262, in getRegionBinRange
    endbin = sorted(self.interval_trees[chrname][endpos:endpos + 1])[0].data
IndexError: list index out of range
ERROR:hicmatrix.HiCMatrix:Index error
Traceback (most recent 

In [7]:
import hicmatrix.HiCMatrix as hm
from hicexplorer import utilities
import scipy.sparse as sp


In [8]:
chic_matrix = hm.hiCMatrix(path_output_cool)

In [9]:
chic_matrix.matrix.shape[0]

273121

In [10]:
pyrange_promoter = pr.read_bed(path_promoter)

In [11]:
pyrange_promoter

,Chromosome,Start,End,Name,Score,Strand,ThickStart,ThickEnd
0,chr1,4807845,4807905,Lypla1_1,900,+,4807894,4807905
1,chr1,4857741,4857801,Tcea1_1,900,+,4857790,4857801
2,chr1,5083079,5083139,Atp6v1h_1,900,+,5083128,5083139
3,chr1,5588420,5588480,Oprk1_1,900,+,5588469,5588480
4,chr1,6214614,6214674,Rb1cc1_1,900,+,6214663,6214674
...,...,...,...,...,...,...,...,...
25106,chrY,1459766,1459826,Usp9y_1,900,-,1459766,1459777
25107,chrY,2170384,2170444,Zfy2_1,900,-,2170384,2170395
25108,chrY,3410158,3410218,Rbmy_1,900,-,3410158,3410169
25109,chrY,89713277,89713337,FJ541083_1,900,-,89713277,89713288


In [12]:
len(pyrange_promoter)

25111

In [13]:
matrix_list = []
bait_list = []
for index, row in pyrange_promoter.as_df().iterrows():
    bait_location = row[['Chromosome', 'Start', 'End']]
    bait_list.append(row[['Name']])
    view_point_start, view_point_end = chic_matrix.getRegionBinRange(bait_location[0], bait_location[1], bait_location[2])

    matrix_list.append(chic_matrix.matrix[view_point_start, :])

# Assuming you have a list of CSR matrices: matrix_list
sample_matrix = sp.vstack(matrix_list)


In [14]:
sample_matrix

<25111x273121 sparse matrix of type '<class 'numpy.float64'>'
	with 167208 stored elements in Compressed Sparse Row format>

In [15]:
import networkx as nx

import community as community_louvain


In [16]:
G = nx.Graph()

bait_ids, OE_ids = sample_matrix.nonzero()
for i, (bait, OE) in enumerate(zip(bait_ids, OE_ids)):
    if (bait != OE):
        G.add_weighted_edges_from([(bait, OE , sample_matrix.data[i])])
        G.nodes[OE]['colorType'] = "orange"


for bait in bait_ids:
    G.nodes[bait]['name'] = bait_list[bait][0]
    G.nodes[bait]['colorType'] = "green"

partition = community_louvain.best_partition(G)

for node, partition_id in partition.items():
    G.nodes[node]['partition'] = partition_id
    G.nodes[node]['partition_color'] = partition_id


nx.write_graphml(G, "graph.graphml")

In [17]:
partition

{1: 0,
 589: 1,
 619: 2,
 1359: 0,
 5079: 0,
 5102: 0,
 5425: 0,
 5530: 0,
 6520: 0,
 8822: 0,
 8825: 0,
 8826: 0,
 8827: 0,
 88463: 0,
 88470: 0,
 112943: 0,
 218774: 0,
 257757: 0,
 2: 3,
 588: 4,
 590: 5,
 4550: 3,
 30445: 3,
 68695: 3,
 138438: 3,
 138439: 3,
 179041: 3,
 210726: 3,
 210727: 6,
 235511: 7,
 3: 8,
 17631: 8,
 21777: 9,
 38732: 8,
 55170: 10,
 72916: 8,
 79753: 8,
 91561: 8,
 93578: 5,
 107387: 8,
 108048: 8,
 127165: 11,
 145828: 8,
 148341: 12,
 170962: 8,
 171040: 8,
 173295: 8,
 240822: 9,
 264426: 8,
 4: 13,
 20308: 13,
 43511: 13,
 62921: 13,
 72282: 87,
 72379: 15,
 77644: 13,
 124260: 13,
 161101: 16,
 186135: 17,
 206800: 13,
 227587: 13,
 6: 15,
 685: 15,
 944: 15,
 3339: 15,
 27936: 15,
 27938: 15,
 33792: 15,
 48965: 15,
 48966: 15,
 71932: 15,
 78793: 15,
 87404: 15,
 146032: 15,
 222830: 18,
 258963: 15,
 9: 19,
 739: 19,
 977: 19,
 978: 19,
 993: 19,
 1001: 19,
 1004: 20,
 1052: 19,
 4129: 19,
 15986: 19,
 15987: 19,
 96318: 19,
 96319: 19,
 104433: 19

In [61]:
max(partition.values())

1334

In [20]:
partition[8766]

6

In [18]:
for bait in bait_list:
    if "Hoxa" in bait[0]:
        print(bait)

Name    Hoxaas2_1
Name: 8138, dtype: object
Name    Hoxaas3_1
Name: 8140, dtype: object
Name    Hoxaas3_2
Name: 8141, dtype: object
Name    Hoxa11os_1
Name: 8142, dtype: object
Name    Hoxa11os_2
Name: 8143, dtype: object
Name    Hoxa1_1
Name: 8766, dtype: object
Name    Hoxa2_1
Name: 8767, dtype: object
Name    Hoxa4_1
Name: 8768, dtype: object
Name    Hoxa6_1
Name: 8769, dtype: object
Name    Hoxa9_1
Name: 8770, dtype: object
Name    Hoxa10_2
Name: 8772, dtype: object
Name    Hoxa10_1
Name: 8773, dtype: object
Name    Hoxa11_1
Name: 8774, dtype: object
Name    Hoxa13_1
Name: 8775, dtype: object


In [21]:
import matplotlib.colors as mcolors

In [22]:
mcolors.Colormap

matplotlib.colors.Colormap

In [23]:
path_histones_H3K3me2 = "/home/joachim/data/MHH/epigenetic_graph_analyzer/histones/GSM2371710_ChIPseq_E10.5-Md_H3K4me2_1.scaled_0100.bw"
path_histones_H3K27me3  = "/home/joachim/data/MHH/epigenetic_graph_analyzer/histones/GSM2371715_ChIPseq_E10.5-Md_H3K27me3_1.scaled_0100.bw"
path_histones_H3K27ac  = "/home/joachim/data/MHH/epigenetic_graph_analyzer/histones/GSM2371719_ChIPseq_E10.5-Md_H3K27ac_1.scaled_0100.bw"

In [20]:
import pyBigWig

In [47]:
chromosome, start, end , _ = chic_matrix.getBinPos(80)

In [49]:
start

800000

In [50]:
import pyBigWig
import gzip

list_of_histones = []
sample, interaction = sample_matrix.nonzero()

for file_path in [path_histones_H3K3me2, path_histones_H3K27me3, path_histones_H3K27ac]:
    # Open the bigwig file
    bw = pyBigWig.open(file_path)

    histone_matrix = dok_matrix(sample_matrix.shape)
    for sample_bin, interaction_bin in zip(sample, interaction):
        chromosome, start, end , _ = chic_matrix.getBinPos(interaction_bin)
        
        # Read the data from the bigwig file
        value = bw.stats(chromosome, start, end)
        histone_matrix[sample_bin, interaction_bin] = value[0]
    
    list_of_histones.append(histone_matrix)

In [76]:
x = sample_matrix.shape[0]
new_dict = {key: value for key, value in partition.items() if key < x}


In [77]:
new_dict

{1: 0,
 589: 1,
 619: 2,
 1359: 0,
 5079: 0,
 5102: 0,
 5425: 0,
 5530: 0,
 6520: 0,
 8822: 0,
 8825: 0,
 8826: 0,
 8827: 0,
 2: 3,
 588: 4,
 590: 5,
 4550: 3,
 3: 8,
 17631: 8,
 21777: 9,
 4: 13,
 20308: 13,
 6: 15,
 685: 15,
 944: 15,
 3339: 15,
 9: 19,
 739: 19,
 977: 19,
 978: 19,
 993: 19,
 1001: 19,
 1004: 20,
 1052: 19,
 4129: 19,
 15986: 19,
 15987: 19,
 10: 19,
 11: 20,
 1008: 20,
 5217: 20,
 10850: 22,
 12: 20,
 13: 25,
 1740: 25,
 3362: 26,
 5458: 25,
 5885: 25,
 14: 25,
 15: 20,
 995: 27,
 1003: 20,
 1007: 20,
 1009: 20,
 1010: 20,
 3475: 20,
 17: 31,
 18: 25,
 974: 25,
 975: 25,
 979: 25,
 7363: 25,
 19: 25,
 20: 3,
 1285: 3,
 7014: 3,
 7015: 3,
 21: 33,
 1277: 33,
 1284: 33,
 23: 34,
 3376: 34,
 7188: 34,
 24: 34,
 25: 37,
 1474: 38,
 1486: 37,
 26: 39,
 1609: 39,
 1667: 40,
 3053: 41,
 19900: 0,
 27: 42,
 1630: 42,
 1642: 42,
 1643: 42,
 4887: 42,
 28: 28,
 1709: 28,
 1773: 28,
 1774: 9,
 5247: 28,
 5499: 43,
 6192: 28,
 6193: 28,
 6487: 27,
 6489: 27,
 6716: 28,
 6717: 

In [79]:
matrix_list = [sample_matrix, *list_of_histones]
# Create matrix per community
community_matrix = np.zeros((max(partition.values()), len(matrix_list)))
for community_id in new_dict:
    # Your code here
    # print(community_id)
    # Update the community_matrix with the desired values
    for i, matrix in enumerate(matrix_list):
        community_matrix[new_dict[community_id], i] = np.mean(matrix[community_id, :].tocsr().data)
    


KeyboardInterrupt: 

In [73]:
np.mean(matrix[community_id, :].tocsr().data)

0.018751868987494782

In [71]:
matrix[community_id, :].tocsr().data

array([0.00911416, 0.02037817, 0.02676329])

In [32]:
bw.chroms()

{'chr1': 195471971,
 'chr10': 130694993,
 'chr11': 122082543,
 'chr12': 120129022,
 'chr13': 120421639,
 'chr14': 124902244,
 'chr15': 104043685,
 'chr16': 98207768,
 'chr17': 94987271,
 'chr18': 90702639,
 'chr19': 61431566,
 'chr1_GL456210_random': 169725,
 'chr1_GL456211_random': 241735,
 'chr1_GL456212_random': 153618,
 'chr1_GL456213_random': 39340,
 'chr1_GL456221_random': 206961,
 'chr2': 182113224,
 'chr3': 160039680,
 'chr4': 156508116,
 'chr4_GL456216_random': 66673,
 'chr4_GL456350_random': 227966,
 'chr4_JH584292_random': 14945,
 'chr4_JH584293_random': 207968,
 'chr4_JH584294_random': 191905,
 'chr4_JH584295_random': 1976,
 'chr5': 151834684,
 'chr5_GL456354_random': 195993,
 'chr5_JH584296_random': 199368,
 'chr5_JH584297_random': 205776,
 'chr5_JH584298_random': 184189,
 'chr5_JH584299_random': 953012,
 'chr6': 149736546,
 'chr7': 145441459,
 'chr7_GL456219_random': 175968,
 'chr8': 129401213,
 'chr9': 124595110,
 'chrM': 16299,
 'chrUn_GL456239': 40056,
 'chrUn_GL456359